# Sparse Attention Patterns

**Prerequisites:** Python basics and the relevant model-serving concepts.  **Estimated time:** 35 minutes.

## Objectives

- Build dense, causal, sliding-window and global-local masks.
- Explain at least one failure boundary and a diagnostic.
- Run deterministic assertions against a small synthetic example.


## Model and assumptions

Compare mask density and discuss which dependencies each pattern removes. All quantities below are deliberately small, deterministic teaching values. The paired local implementations are dependency-light (the MoE example uses NumPy) and do not call a model or network service. A synthetic proxy establishes accounting behavior, not production quality or speed.


In [ ]:
# Load the paired standalone implementation from the repository root.
import runpy
impl = runpy.run_path('modern-ai/implementations/30-sparse-attention.py')


## Transparent baseline

Inspect the short implementation above: the critical state and accounting are visible, so you can change one assumption at a time. In a real system, replace the toy inputs with representative traces and validate quality as well as resource use.


In [ ]:
m=impl['mask'](8,'sliding',window=2); g=impl['mask'](8,'global_local',window=2,global_indices=(0,))
print('sliding density',impl['density'](m),'global-local density',impl['density'](g))
assert all(not m[q][k] for q in range(8) for k in range(8) if k>q or k<q-2)


## Comparison and metric

Use the printed counts, mask density, error, latency distribution, or decision labels to compare the alternatives. The selected metric is a proxy for a specific resource or behavior; it is not a substitute for end-to-end evaluation.

In [ ]:
# Edge case: reject invalid inputs or expose degraded behavior instead of hiding it.
try:
    impl['mask'](4,'sliding',window=-1)
except (ValueError, MemoryError, IndexError) as exc:
    print(type(exc).__name__, exc)


## Failure cases and exercise

1. Change the topic parameter that exercises a boundary. Predict which metric changes before running the cell.
2. Identify a case where the proxy gives a misleading decision. State what real trace or quality label you would collect before deployment.

**Exercise:** Use the invariant below as a starting test, then add a second case that probes a boundary relevant to your workload.

In [ ]:
local=impl['mask'](8,'sliding',2); dense=impl['mask'](8,'dense')
assert sum(map(sum,local)) < sum(map(sum,dense))


## Summary and next lessons

This notebook made one sparse attention patterns mechanism inspectable with deterministic accounting. Revisit the concept note at `../concepts/30-sparse-attention.md`, then compare with the neighboring serving lessons and evaluate on a representative workload before making a system choice.
